# 02 — Exploratory Data Analysis (EDA)
**Logistics Operations Analytics Dashboard**

---

This notebook performs deep exploratory analysis:
1. Univariate distributions
2. Bivariate relationships
3. Temporal trends (daily, weekly, monthly)
4. City-wise analysis
5. Correlation heatmap
6. Key insights summary

In [ ]:
# ─── Imports ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

# ─── Premium Color Palette (Porter.in inspired) ──────────────────────────────
PALETTE = {
    'primary': '#2563eb',
    'primary_light': '#60a5fa',
    'success': '#10b981',
    'warning': '#f59e0b',
    'danger': '#ef4444',
    'purple': '#8b5cf6',
    'pink': '#ec4899',
    'teal': '#14b8a6',
    'dark': '#0f172a',
    'gray': '#64748b'
}
COLOR_SEQ = ['#2563eb', '#10b981', '#f59e0b', '#ef4444', '#8b5cf6',
             '#ec4899', '#14b8a6', '#60a5fa', '#f97316', '#6366f1']

# Set global style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.facecolor': '#fafbfc',
    'axes.facecolor': '#fafbfc',
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.labelsize': 12
})

print('Libraries loaded. Ready for EDA!')

In [ ]:
# Load cleaned data (or raw if cleaned not available)
import os
cleaned_path = '../data/logistics_orders_cleaned.csv'
raw_path = '../data/logistics_orders.csv'

if os.path.exists(cleaned_path):
    df = pd.read_csv(cleaned_path)
    print(f'Loaded CLEANED dataset: {df.shape}')
else:
    df = pd.read_csv(raw_path)
    print(f'Loaded RAW dataset: {df.shape}')

# Parse dates
df['order_date'] = pd.to_datetime(df['order_date'])
df['order_hour'] = pd.to_datetime(df['order_time'], format='%H:%M:%S').dt.hour
df['month'] = df['order_date'].dt.to_period('M')
df['month_str'] = df['order_date'].dt.strftime('%Y-%m')
df['day_of_week'] = df['order_date'].dt.day_name()
df['is_weekend'] = df['order_date'].dt.dayofweek >= 5

# Derived columns
df['delivery_delay'] = df['actual_delivery_min'] - df['estimated_delivery_min']
df['is_delayed'] = df['delivery_delay'] > 0

df.head()

---
## 1. Univariate Distributions

In [ ]:
# ─── Order Value Distribution ────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. Order Value
axes[0,0].hist(df['order_value'].dropna(), bins=60, color=PALETTE['primary'],
               alpha=0.8, edgecolor='white', linewidth=0.5)
axes[0,0].axvline(df['order_value'].median(), color=PALETTE['danger'], linestyle='--',
                  label=f'Median: Rs.{df["order_value"].median():,.0f}')
axes[0,0].set_title('Order Value Distribution', fontweight='bold')
axes[0,0].set_xlabel('Order Value (INR)')
axes[0,0].legend()

# 2. Delivery Time
delivered = df[df['order_status'] == 'Delivered']
axes[0,1].hist(delivered['actual_delivery_min'].dropna(), bins=50,
               color=PALETTE['success'], alpha=0.8, edgecolor='white')
axes[0,1].axvline(delivered['actual_delivery_min'].median(), color=PALETTE['danger'],
                  linestyle='--', label=f'Median: {delivered["actual_delivery_min"].median():.0f} min')
axes[0,1].set_title('Delivery Time Distribution', fontweight='bold')
axes[0,1].set_xlabel('Minutes')
axes[0,1].legend()

# 3. Distance
axes[0,2].hist(df['distance_km'].dropna(), bins=50, color=PALETTE['purple'],
               alpha=0.8, edgecolor='white')
axes[0,2].set_title('Distance Distribution', fontweight='bold')
axes[0,2].set_xlabel('Distance (km)')

# 4. Rating
rating_counts = df['rating'].dropna().value_counts().sort_index()
axes[1,0].bar(rating_counts.index.astype(str), rating_counts.values,
              color=PALETTE['warning'], alpha=0.8, edgecolor='white')
axes[1,0].set_title('Rating Distribution', fontweight='bold')
axes[1,0].set_xlabel('Rating')

# 5. Vehicle Type
vt_counts = df['vehicle_type'].value_counts()
bars = axes[1,1].bar(vt_counts.index, vt_counts.values, color=COLOR_SEQ[:4],
                     alpha=0.85, edgecolor='white')
axes[1,1].bar_label(bars, fmt='{:,.0f}', fontsize=10)
axes[1,1].set_title('Orders by Vehicle Type', fontweight='bold')

# 6. Order Status
status_counts = df['order_status'].value_counts()
colors_status = [PALETTE['success'], PALETTE['danger'], PALETTE['warning'], PALETTE['purple']]
wedges, texts, autotexts = axes[1,2].pie(
    status_counts.values, labels=status_counts.index,
    autopct='%1.1f%%', colors=colors_status, startangle=90,
    textprops={'fontsize': 10}
)
axes[1,2].set_title('Order Status Split', fontweight='bold')

fig.suptitle('Univariate Distributions', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../data/eda_univariate.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 2. Bivariate Relationships

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Revenue by City (Top 10)
city_rev = df.groupby('city')['total_revenue'].sum().sort_values(ascending=True).tail(10)
bars = axes[0,0].barh(city_rev.index, city_rev.values / 1e5,
                      color=PALETTE['primary'], alpha=0.85, edgecolor='white')
axes[0,0].set_xlabel('Revenue (Lakhs INR)')
axes[0,0].set_title('Top 10 Cities by Revenue', fontweight='bold')
axes[0,0].bar_label(bars, fmt='%.1fL', padding=5, fontsize=9)

# 2. Delivery Time vs Distance (scatter)
sample = delivered.sample(min(3000, len(delivered)), random_state=42)
scatter = axes[0,1].scatter(sample['distance_km'], sample['actual_delivery_min'],
                            c=sample['rating'].fillna(3), cmap='RdYlGn',
                            alpha=0.4, s=15, edgecolors='none')
plt.colorbar(scatter, ax=axes[0,1], label='Rating')
axes[0,1].set_xlabel('Distance (km)')
axes[0,1].set_ylabel('Delivery Time (min)')
axes[0,1].set_title('Delivery Time vs Distance', fontweight='bold')

# 3. Cancellation Rate by Vehicle Type
cancel_by_vt = df.groupby('vehicle_type').apply(
    lambda x: (x['order_status'] == 'Cancelled').mean() * 100
).sort_values(ascending=True)
bars = axes[1,0].barh(cancel_by_vt.index, cancel_by_vt.values,
                      color=PALETTE['danger'], alpha=0.8, edgecolor='white')
axes[1,0].set_xlabel('Cancellation Rate (%)')
axes[1,0].set_title('Cancellation Rate by Vehicle Type', fontweight='bold')
axes[1,0].bar_label(bars, fmt='%.1f%%', padding=5)

# 4. Average Order Value by Payment Mode
aov_payment = df.groupby('payment_mode')['order_value'].mean().sort_values(ascending=True)
bars = axes[1,1].barh(aov_payment.index, aov_payment.values,
                      color=COLOR_SEQ[:4], alpha=0.85, edgecolor='white')
axes[1,1].set_xlabel('Average Order Value (INR)')
axes[1,1].set_title('AOV by Payment Mode', fontweight='bold')
axes[1,1].bar_label(bars, fmt='Rs.{:,.0f}', padding=5)

fig.suptitle('Bivariate Analysis', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../data/eda_bivariate.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. Temporal Trends

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 10))

# 1. Monthly Revenue Trend
monthly = df[df['order_status'] != 'Cancelled'].groupby('month_str').agg(
    revenue=('total_revenue', 'sum'),
    orders=('order_id', 'count')
).reset_index()

ax1 = axes[0,0]
ax1.fill_between(range(len(monthly)), monthly['revenue'] / 1e5,
                 alpha=0.3, color=PALETTE['primary'])
ax1.plot(range(len(monthly)), monthly['revenue'] / 1e5,
         color=PALETTE['primary'], linewidth=2.5, marker='o', markersize=4)
ax1.set_xticks(range(0, len(monthly), 3))
ax1.set_xticklabels(monthly['month_str'].iloc[::3], rotation=45, ha='right')
ax1.set_ylabel('Revenue (Lakhs INR)')
ax1.set_title('Monthly Revenue Trend', fontweight='bold')

# 2. Daily Order Volume (rolling 7-day avg)
daily = df.groupby('order_date')['order_id'].count().reset_index()
daily.columns = ['date', 'orders']
daily['rolling_7d'] = daily['orders'].rolling(7).mean()

ax2 = axes[0,1]
ax2.plot(daily['date'], daily['orders'], alpha=0.25, color=PALETTE['gray'], linewidth=0.5)
ax2.plot(daily['date'], daily['rolling_7d'], color=PALETTE['success'], linewidth=2,
         label='7-day Rolling Avg')
ax2.set_title('Daily Order Volume', fontweight='bold')
ax2.set_ylabel('Orders')
ax2.legend()
ax2.xaxis.set_major_locator(mdates.MonthLocator(interval=4))
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha='right')

# 3. Peak Order Hours
hourly = df.groupby('order_hour')['order_id'].count()
ax3 = axes[1,0]
colors_hour = [PALETTE['primary'] if h in range(11, 14) or h in range(18, 22)
               else PALETTE['gray'] for h in range(24)]
bars = ax3.bar(hourly.index, hourly.values, color=colors_hour,
               alpha=0.85, edgecolor='white', linewidth=0.5)
ax3.set_xlabel('Hour of Day')
ax3.set_ylabel('Order Count')
ax3.set_title('Orders by Hour (Peak Hours Highlighted)', fontweight='bold')
ax3.set_xticks(range(0, 24, 2))

# 4. Day of Week Analysis
dow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow_data = df.groupby('day_of_week').agg(
    orders=('order_id', 'count'),
    revenue=('total_revenue', 'sum')
).reindex(dow_order)

ax4 = axes[1,1]
ax4_twin = ax4.twinx()
bars = ax4.bar(range(7), dow_data['orders'], color=PALETTE['primary'],
               alpha=0.6, edgecolor='white', label='Orders')
ax4_twin.plot(range(7), dow_data['revenue'] / 1e5, color=PALETTE['danger'],
              linewidth=2.5, marker='s', markersize=6, label='Revenue')
ax4.set_xticks(range(7))
ax4.set_xticklabels([d[:3] for d in dow_order])
ax4.set_ylabel('Orders', color=PALETTE['primary'])
ax4_twin.set_ylabel('Revenue (Lakhs)', color=PALETTE['danger'])
ax4.set_title('Orders & Revenue by Day of Week', fontweight='bold')

fig.suptitle('Temporal Trends', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../data/eda_temporal.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. City-Wise Deep Dive

In [ ]:
# City performance matrix
city_perf = df.groupby('city').agg(
    total_orders=('order_id', 'count'),
    total_revenue=('total_revenue', 'sum'),
    avg_order_value=('order_value', 'mean'),
    avg_delivery_time=('actual_delivery_min', 'mean'),
    avg_rating=('rating', 'mean'),
    cancel_rate=('order_status', lambda x: (x == 'Cancelled').mean() * 100),
    unique_customers=('customer_id', 'nunique'),
    unique_drivers=('driver_id', 'nunique')
).round(2).sort_values('total_revenue', ascending=False)

city_perf['revenue_lakhs'] = (city_perf['total_revenue'] / 1e5).round(2)
print('City Performance Matrix (Top 15):')
city_perf.head(15)

In [ ]:
# City comparison heatmap
top_cities = city_perf.head(10)[['total_orders', 'revenue_lakhs', 'avg_order_value',
                                  'avg_delivery_time', 'avg_rating', 'cancel_rate']]

# Normalize for heatmap
top_normalized = (top_cities - top_cities.min()) / (top_cities.max() - top_cities.min())

fig, ax = plt.subplots(figsize=(14, 7))
sns.heatmap(top_normalized, annot=top_cities.values, fmt='.1f', cmap='RdYlGn',
            linewidths=1, linecolor='white', ax=ax, cbar_kws={'label': 'Normalized Score'})
ax.set_title('Top 10 Cities — Performance Heatmap', fontsize=14, fontweight='bold')
ax.set_xticklabels(['Orders', 'Revenue (L)', 'AOV', 'Delivery Time', 'Rating', 'Cancel %'],
                   rotation=30, ha='right')
plt.tight_layout()
plt.savefig('../data/eda_city_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Correlation Heatmap

In [ ]:
# Correlation matrix
numeric_cols = ['order_value', 'delivery_fee', 'total_revenue', 'distance_km',
                'estimated_delivery_min', 'actual_delivery_min', 'rating']
corr = df[numeric_cols].corr().round(2)

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, cmap='coolwarm', center=0,
            square=True, linewidths=1, linecolor='white', ax=ax,
            fmt='.2f', vmin=-1, vmax=1,
            cbar_kws={'shrink': 0.8, 'label': 'Correlation'})
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/eda_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Additional Insights

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Cancellation Reasons (Top 8)
cancel_reasons = df[df['order_status'] == 'Cancelled']['cancellation_reason'].value_counts().head(8)
bars = axes[0].barh(range(len(cancel_reasons)), cancel_reasons.values,
                    color=PALETTE['danger'], alpha=0.8, edgecolor='white')
axes[0].set_yticks(range(len(cancel_reasons)))
axes[0].set_yticklabels([r[:35] for r in cancel_reasons.index], fontsize=9)
axes[0].set_title('Top Cancellation Reasons', fontweight='bold')
axes[0].bar_label(bars, padding=5, fontsize=9)

# 2. Payment Mode Share
pm_counts = df['payment_mode'].value_counts()
pm_colors = [PALETTE['primary'], PALETTE['success'], PALETTE['warning'], PALETTE['purple']]
axes[1].pie(pm_counts.values, labels=pm_counts.index, autopct='%1.1f%%',
            colors=pm_colors, startangle=90, textprops={'fontsize': 11},
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Payment Mode Distribution', fontweight='bold')

# 3. Repeat vs New Customers
repeat = df.groupby('is_repeat_customer').agg(
    orders=('order_id', 'count'),
    revenue=('total_revenue', 'sum')
)
x = np.arange(2)
w = 0.35
bars1 = axes[2].bar(x - w/2, repeat['orders'], w, label='Orders',
                    color=PALETTE['primary'], alpha=0.8)
ax2 = axes[2].twinx()
bars2 = ax2.bar(x + w/2, repeat['revenue'] / 1e5, w, label='Revenue (L)',
                color=PALETTE['success'], alpha=0.8)
axes[2].set_xticks(x)
axes[2].set_xticklabels(['New', 'Repeat'])
axes[2].set_ylabel('Orders', color=PALETTE['primary'])
ax2.set_ylabel('Revenue (Lakhs)', color=PALETTE['success'])
axes[2].set_title('New vs Repeat Customers', fontweight='bold')
axes[2].legend(loc='upper left')
ax2.legend(loc='upper right')

plt.tight_layout()
plt.savefig('../data/eda_insights.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Key Findings Summary

In [ ]:
print('='*60)
print('KEY EDA FINDINGS')
print('='*60)

total_rev = df['total_revenue'].sum()
total_orders = len(df)
cancel_rate = (df['order_status'] == 'Cancelled').mean() * 100
avg_delivery = delivered['actual_delivery_min'].mean()
top_city = df.groupby('city')['total_revenue'].sum().idxmax()
peak_hour = df.groupby('order_hour')['order_id'].count().idxmax()
top_payment = df['payment_mode'].mode()[0]
repeat_pct = df['is_repeat_customer'].mean() * 100

findings = [
    f'Total Revenue: Rs.{total_rev:,.0f} ({total_rev/1e7:.1f} Cr)',
    f'Total Orders: {total_orders:,}',
    f'Average Order Value: Rs.{df["order_value"].mean():,.0f}',
    f'Cancellation Rate: {cancel_rate:.1f}%',
    f'Average Delivery Time: {avg_delivery:.1f} minutes',
    f'Top Revenue City: {top_city}',
    f'Peak Order Hour: {peak_hour}:00',
    f'Most Popular Payment: {top_payment}',
    f'Repeat Customer Rate: {repeat_pct:.1f}%',
    f'Cities Covered: {df["city"].nunique()}',
    f'Active Drivers: {df["driver_id"].nunique()}',
    f'Unique Customers: {df["customer_id"].nunique():,}'
]

for i, f in enumerate(findings, 1):
    print(f'  {i:2d}. {f}')

print('\nEDA Complete!')